In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2025-01")
v_file_date = dbutils.widgets.get("p_file_date")
raw_race_path = f"{raw_folder_path}/{v_file_date}"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import current_timestamp

In [0]:
if USE_INCREMENTAL:
    dbutils.notebook.exit("Skipped: qualifying not present in incremental drops")
    
qualifying_schema = StructType([
  StructField("qualifyId", IntegerType(), False),
  StructField("raceId", IntegerType(), False),
  StructField("driverId", IntegerType(), False),
  StructField("constructorId", IntegerType(), False),
  StructField("number", IntegerType(), False),
  StructField("position", IntegerType(), True),
  StructField("q1", StringType(), True),
  StructField("q2", StringType(), True),
  StructField("q3", StringType(), True),
])

qualifying_df = spark.read \
  .schema(qualifying_schema) \
  .option("multiline", True) \
  .json(f"{raw_race_path}/qualifying/qualifying_split*.json")
display(qualifying_df)

In [0]:
final_qualifying_df = qualifying_df \
  .withColumnRenamed("qualifyId", "qualifying_id") \
  .withColumnRenamed("raceId", "race_id") \
  .withColumnRenamed("driverId", "driver_id") \
  .withColumnRenamed("constructorId", "constructor_id") \
  .withColumnRenamed("driverId", "driver_id") \
  .withColumn("ingestion_date", current_timestamp())
display(final_qualifying_df)

In [0]:
final_qualifying_df.write.mode("overwrite").parquet(f"{processed_folder_path}/qualifying")

In [0]:
df = spark.read.parquet(f"{processed_folder_path}/qualifying")
display(df)

In [0]:
dbutils.notebook.exit("Success")